# 03. Thuật Toán SKNN (Session K-Nearest Neighbors)

## Ý tưởng
1. Người dùng đang click: [A, B, C]
2. Tìm trong lịch sử: phiên nào giống nhất?
3. Phiên giống nhất click gì tiếp? → Gợi ý cái đó!

## Công thức
- **Similarity:** `sim(Sq, Sn) = |Sq ∩ Sn| / √(|Sq| × |Sn|)`
- **Score:** `score(i) = Σ sim(Sq, Sn) × 1[i ∈ Sn]` (với n ∈ KNN)

In [ ]:
# --- Cho phép import gói src/ (notebook đặt ở thư mục gốc dự án) ---
import sys
from pathlib import Path
_root = Path.cwd()
if not (_root / "src").is_dir() and (_root.parent / "src").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import math

from src import config
from src.models import SKNN, PopularityBaseline
from src.evaluate import evaluate
from src.data import load_processed

print("Gói src/ đã sẵn sàng (SKNN, PopularityBaseline, evaluate import từ src)")


## 3.1. Thuật toán SKNN

Lớp `SKNN` được cài trong `src/models/sknn.py` (dùng inverted index để chạy nhanh). Ở đây ta import và sử dụng, không định nghĩa lại.

- **Similarity:** `sim(Sq, Sn) = |Sq ∩ Sn| / √(|Sq| × |Sn|)`
- **Score:** `score(i) = Σ sim(Sq, Sn)` với `n ∈ KNN` và `i ∈ Sn`

## 3.2. Ví dụ tính tay (kiểm tra thuật toán đúng)

In [ ]:
# Dữ liệu mẫu nhỏ
sample_train = {
    'S1': ['I1', 'I2', 'I3', 'I4'],
    'S2': ['I1', 'I2', 'I5'],
    'S3': ['I2', 'I3', 'I6'],
    'S4': ['I5', 'I6', 'I7'],
    'S5': ['I1', 'I3', 'I4', 'I8'],
}
sample_query = ['I1', 'I2', 'I3']

print('=== VÍ DỤ TÍNH TAY ===')
print(f'Phiên truy vấn: {sample_query}')
print()

# Tính similarity thủ công
model_sample = SKNN(k=3)
model_sample.fit(sample_train)

print()
print('Tính similarity từng phiên:')
for sid, items in sample_train.items():
    sq_set = set(sample_query)
    sn_set = set(items)
    inter = sq_set & sn_set
    sim = len(inter) / math.sqrt(len(sq_set) * len(sn_set)) if inter else 0
    print(f'  sim(Query, {sid}) = |{inter}| / √({len(sq_set)}×{len(sn_set)}) = {sim:.3f}')

print()
print('Top-3 khuyến nghị:')
recs = model_sample.predict(sample_query, top_n=3)
for rank, (item, score) in enumerate(recs, 1):
    print(f'  #{rank}: {item} (score = {score:.3f})')

print()
print('→ I4 được gợi ý đầu tiên vì phiên S1 (giống nhất) có chứa I4')
print('→ Rất hợp lý: người đang đọc sách kỹ năng mềm → gợi ý sách kỹ năng mềm khác')

## 3.3. Popularity Baseline (để so sánh)

Lớp `PopularityBaseline` nằm trong `src/models/popularity.py` — gợi ý item phổ biến nhất, bỏ qua thứ tự phiên.

## 3.4. Hàm đánh giá

Hàm `evaluate()` (giao thức leave-one-out, Recall@N / MRR@N) nằm trong `src/evaluate.py`.

## 3.5. Chạy trên dữ liệu thật

In [ ]:
# Load dữ liệu đã xử lý từ notebook 02 (hoặc dùng mẫu nhỏ để demo)
if config.PROCESSED_PATH.exists():
    train_sessions, test_sessions = load_processed(config.PROCESSED_PATH)
    print(f"✓ Đã load dữ liệu: {len(train_sessions):,} train, {len(test_sessions):,} test")
else:
    print("⚠ Chưa có dữ liệu thật. Dùng dữ liệu mẫu để demo.")
    train_sessions = sample_train
    test_sessions = {
        'T1': ['I1','I2','I3','I4'], 'T2': ['I2','I3','I6'],
        'T3': ['I5','I6','I7'], 'T4': ['I1','I3','I8'],
    }


In [ ]:
print('=' * 55); print('ĐÁNH GIÁ SKNN'); print('=' * 55)
model_sknn = SKNN(k=config.SKNN_K)
model_sknn.fit(train_sessions)
print('\nĐang đánh giá trên TOÀN BỘ tập test...')
r_sknn, mrr_sknn, n = evaluate(model_sknn, test_sessions, top_n=config.TOP_N, max_eval=None)
print(f'\n  Recall@20 = {r_sknn:.4f} ({r_sknn*100:.2f}%)')
print(f'  MRR@20    = {mrr_sknn:.4f}')
print(f'  Số phiên đánh giá: {n:,}')


In [ ]:
print('=' * 55); print('ĐÁNH GIÁ POPULARITY BASELINE'); print('=' * 55)
model_pop = PopularityBaseline()
model_pop.fit(train_sessions)
print('\nĐang đánh giá trên TOÀN BỘ tập test...')
r_pop, mrr_pop, _ = evaluate(model_pop, test_sessions, top_n=config.TOP_N, max_eval=None)
print(f'\n  Recall@20 = {r_pop:.4f} ({r_pop*100:.2f}%)')
print(f'  MRR@20    = {mrr_pop:.4f}')


In [ ]:
print('\n' + '=' * 55); print('  BẢNG SO SÁNH'); print('=' * 55)
print(f'  {"Mô hình":<25} {"Recall@20":>10} {"MRR@20":>8}')
print(f'  {"-"*45}')
print(f'  {"Popularity Baseline":<25} {r_pop:>10.4f} {mrr_pop:>8.4f}')
print(f'  {"SKNN (k="+str(config.SKNN_K)+")":<25} {r_sknn:>10.4f} {mrr_sknn:>8.4f}')
print(f'  {"Cải thiện":<25} {r_sknn-r_pop:>+10.4f} {mrr_sknn-mrr_pop:>+8.4f}')
print('=' * 55)
print(f'\n→ SKNN tốt hơn Popularity {(r_sknn-r_pop)*100:.1f}% Recall@20')

# Lưu kết quả vào output/all_results.pkl để notebook 05 đọc (không hardcode số)
import pickle
config.OUTPUT_DIR.mkdir(exist_ok=True)
all_results = {}
if config.RESULTS_PATH.exists():
    with open(config.RESULTS_PATH, 'rb') as f:
        all_results = pickle.load(f)
all_results['popularity'] = {'recall': r_pop, 'mrr': mrr_pop}
all_results['sknn'] = {'recall': r_sknn, 'mrr': mrr_sknn}
all_results.setdefault('meta', {})['n_eval'] = n
with open(config.RESULTS_PATH, 'wb') as f:
    pickle.dump(all_results, f)
print('✓ Đã lưu popularity + sknn vào', config.RESULTS_PATH)
